# Day 1 Corpus Generation — 3 seeds × 500 trajectories (H100)

Tier 1 expansion: with H100 throughput (~5–6s/traj), we can afford 3 RNG seeds at 500 trajectories each (1500 total) inside ~1 hour wall clock.

Each seed writes to its own subdir under `data/`:
- `data/seed_42/trajectories.jsonl` + `data/seed_42/activations/`
- `data/seed_43/trajectories.jsonl` + `data/seed_43/activations/`
- `data/seed_44/trajectories.jsonl` + `data/seed_44/activations/`

The catalog (`data/catalog.json`) is shared across seeds — only the query/perturbation RNG varies. This is the right ablation: same world, different episodes.

Workflow:
1. Mount Drive, clone/pull repo, install deps.
2. Launch all 3 seed runs in parallel as background processes (Qwen2.5-7B in bf16 is ~14 GB; 3 copies fit comfortably on an 80 GB H100).
3. Watch progress across all three until at least one is past 20 trajectories.
4. Resume on disconnect via `--resume` flag (already set).

## 1. Mount + clone + install

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_ROOT = '/content/drive/MyDrive/agent_faithfulness'
REPO_DIR  = '/content/agent_faithfulness'
GITHUB_URL = 'https://github.com/lebretou/agent_faithfulness.git'

import os, subprocess
os.makedirs(f'{DRIVE_ROOT}/data', exist_ok=True)
os.makedirs(f'{DRIVE_ROOT}/logs', exist_ok=True)

if not os.path.exists(REPO_DIR):
    subprocess.check_call(['git', 'clone', GITHUB_URL, REPO_DIR])
else:
    subprocess.check_call(['git', '-C', REPO_DIR, 'pull'])

%cd $REPO_DIR
!pip install -q -r requirements.txt

## 2. Launch 3 seed runs in parallel

Each is a separate `nohup python scripts/01_generate_corpus.py ... &` writing to its own seed directory and log file.

In [ ]:
import time, os

TIMESTAMP = int(time.time())
SEEDS = [42, 43, 44]
N_TRAJ = 500

for seed in SEEDS:
    out_dir = f'{DRIVE_ROOT}/data/seed_{seed}'
    log_path = f'{DRIVE_ROOT}/logs/gen_seed{seed}_{TIMESTAMP}.log'
    cmd = (
        f'cd {REPO_DIR} && nohup python scripts/01_generate_corpus.py '
        f'--config configs/default.yaml '
        f'--n_trajectories {N_TRAJ} '
        f'--out_dir {out_dir} '
        f'--seed {seed} '
        f'--resume '
        f'> {log_path} 2>&1 &'
    )
    print(f'[launch seed={seed}] log={log_path}')
    os.system(cmd)
    time.sleep(2)  # stagger model load slightly to avoid simultaneous HF download

print('\nAll 3 jobs launched.')
!sleep 5 && pgrep -af 01_generate_corpus.py

## 3. Watch all three until one is past 20 trajectories

In [ ]:
import time
from pathlib import Path

JSONLS = {seed: Path(f'{DRIVE_ROOT}/data/seed_{seed}/trajectories.jsonl') for seed in SEEDS}

for tick in range(60):  # ~60 minutes of observation max
    counts = {}
    for seed, p in JSONLS.items():
        n = 0
        if p.exists():
            with open(p) as f:
                n = sum(1 for _ in f)
        counts[seed] = n
    line = '  '.join(f'seed_{s}={n}' for s, n in counts.items())
    print(f't+{int(time.time())%100000:5d}  {line}')
    if max(counts.values()) >= 20:
        print('At least one seed past 20 trajectories — rate confirmed. Walk away.')
        break
    time.sleep(60)

## 4. Health check (run any time)

In [ ]:
# Are all 3 still alive? Counts on disk?
!pgrep -af 01_generate_corpus.py || echo 'NO PROCESSES RUNNING'

import time
from pathlib import Path
for seed in SEEDS:
    p = Path(f'{DRIVE_ROOT}/data/seed_{seed}/trajectories.jsonl')
    n = 0
    mtime = '<missing>'
    if p.exists():
        with open(p) as f:
            n = sum(1 for _ in f)
        mtime = time.ctime(p.stat().st_mtime)
    print(f'seed_{seed}: {n:4d} trajectories  last_write={mtime}')

# Tail latest log
!ls -t {DRIVE_ROOT}/logs/gen_seed*.log | head -3 | xargs -I{} sh -c 'echo "--- {} ---"; tail -n 8 {}'

In [ ]:
# (Optional) Kill all runs.
# !pkill -f 01_generate_corpus.py